In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import uproot
import glob
import awkward as ak
import itertools
import yaml
import os
import sys
from tqdm import tqdm
from pathlib import Path
import atlasify as atl
atl.ATLAS = "ColliderML"

sys.path.append("../")
import pyhepmc as hep
from pyhepmc.io import WriterAscii

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
# Configure paths and expected parameters
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

merged_path = Path("/global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_pilot/ttbar/v4/runs/0/merged_events.hepmc3")
assert merged_path.exists(), f"Merged file not found: {merged_path}"

mu = 200.0
print("Merged file:", merged_path)
print("Expected Poisson mu:", mu)


Merged file: /global/cfs/cdirs/m4958/data/ColliderML/simulation/full_pileup_pilot/ttbar/v4/runs/0/merged_events.hepmc3
Expected Poisson mu: 200.0


In [20]:
evt.numpy.vertices.status

array([0, 0, 0, ..., 0, 0, 0], shape=(50272,), dtype=int32)

In [27]:
(evt.numpy.particles.status == 4).sum() / 2

np.float64(204.0)

In [ ]:
# Fast beam-particle method: k = (num_beam_particles / 2) - 1 per event
from pathlib import Path
import numpy as np

path = merged_path
k_beam = []
beam = 0

with path.open("r") as f:
    for line in f:
        if line.startswith("E "):
            if beam or not k_beam:
                k_beam.append(max(0, beam // 2 - 1))
            beam = 0
        elif line.startswith("P "):
            # HepMC3 ASCII v3: last token is status; 4 denotes incoming beam
            # Use rsplit to be robust to scientific notation in earlier fields
            if line.rsplit(maxsplit=1)[-1].strip() == "4":
                beam += 1
# last event
if beam or not k_beam:
    k_beam.append(max(0, beam // 2 - 1))

k = np.asarray(k_beam, dtype=int)
print(f"[beam method] events={len(k)} mean={k.mean():.2f} std={k.std(ddof=1):.2f}")


In [9]:
# Use pyhepmc to read the merged events and estimate pileup multiplicity
import pyhepmc as hep

multiplicities = []
events_to_read = 3

with hep.open(str(merged_path)) as f:
    for i, evt in tqdm(enumerate(f)):
        # Count primary vertices as those with no incoming particles
        primaries = sum(1 for v in evt.vertices if len(v.particles_in) == 0)
        multiplicities.append(primaries)
        break
        if i >= events_to_read:
            break

k = np.asarray(multiplicities, dtype=int) - 1
k = k[k >= 0]
print(f"Read {len(k)} events. k mean={k.mean():.2f}, std={k.std(ddof=1):.2f}")


0it [00:25, ?it/s]

Read 0 events. k mean=nan, std=nan



/tmp/ipykernel_532230/385195753.py:18: RuntimeWarning: Mean of empty slice.
  print(f"Read {len(k)} events. k mean={k.mean():.2f}, std={k.std(ddof=1):.2f}")
/global/homes/d/danieltm/.conda/envs/collider-env/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/global/homes/d/danieltm/.conda/envs/collider-env/lib/python3.11/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/global/homes/d/danieltm/.conda/envs/collider-env/lib/python3.11/site-packages/numpy/_core/_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/global/homes/d/danieltm/.conda/envs/collider-env/lib/python3.11/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
evt.vertices

[GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73e+03)),
 GenVertex(FourVector(0.0154, 0.018, 4.54, -1.73

In [ ]:
# Plot histogram and compare to Poisson(mu)
import scipy.stats as st

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

if len(k):
    bins = np.arange(max(1, int(mu - 5*np.sqrt(mu))), int(mu + 5*np.sqrt(mu)) + 2)
    ax[0].hist(k, bins=bins, alpha=0.6, label="empirical", density=True)
    x = np.arange(bins[0], bins[-1])
    ax[0].plot(x, st.poisson(mu).pmf(x), 'r-', lw=1.0, label=f"Poisson($\\mu$={mu:.1f})")
    ax[0].legend()
    ax[0].set_ylabel("density")
ax[0].set_xlabel("k (pileup multiplicity)")
ax[0].set_title("Pileup multiplicity distribution")

emp_mean = float(np.mean(k)) if len(k) else float('nan')
emp_var = float(np.var(k, ddof=1)) if len(k) > 1 else float('nan')
print(f"Empirical mean={emp_mean:.2f}, var={emp_var:.2f} (expected mean=var≈{mu:.2f})")

if len(k) >= 10:
    empirical = np.sort(k)
    quantiles = np.linspace(0.01, 0.99, 50)
    q_emp = np.quantile(empirical, quantiles)
    q_pois = st.poisson(mu).ppf(quantiles)
    ax[1].plot(q_pois, q_emp, 'o', ms=3)
    ax[1].plot([q_pois.min(), q_pois.max()], [q_pois.min(), q_pois.max()], 'k--', lw=1)
    ax[1].set_title("Quantile-Quantile plot")
ax[1].set_xlabel("Poisson quantiles")
ax[1].set_ylabel("Empirical quantiles")
plt.tight_layout(); plt.show()
